# PepCABO Evaluation

This notebook loads the trained models and compares random selection against guided search for peptide recommendation.


In [ ]:
import contextlib
import os
import random
import sys
import warnings
from pathlib import Path

import gpytorch
import mhcflurry
import numpy as np
import pandas as pd
import pytorch_lightning as pl
import torch
from IPython.display import display
from linear_operator.utils.cholesky import NumericalWarning
from scipy.stats import percentileofscore
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

sys.path.append('..')

from pepcabo.utils.bo_utils.ppgpr import GPModelDKLExtended
from pepcabo.utils.bo_utils.turbo import TS, TurboState, generate_batch
from pepcabo.utils.pep_utils.seq_vae.JointTraining import JointTraining
from pepcabo.utils.pep_utils.seq_vae.data import AlleleDataset, PairsDataModule, PeptideDataset
from pepcabo.utils.pep_utils.seq_vae.model_allele_ae import InfoCNNVAE2
from pepcabo.utils.pep_utils.seq_vae.model_positional_unbounded import InfoTransformerVAE

warnings.filterwarnings('ignore', category=NumericalWarning)

ROOT = Path('..')
DATA_DIR = ROOT / 'data'
MODEL_DIR = DATA_DIR / 'models'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


@contextlib.contextmanager
def suppress_stdout():
    with open(os.devnull, 'w') as devnull:
        old_stdout = sys.stdout
        old_stderr = sys.stderr
        try:
            sys.stdout = devnull
            sys.stderr = devnull
            yield
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr


def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True


def percentile_mean(reference, values):
    return np.mean([percentileofscore(reference, value, 'rank') for value in values])


def affinity_to_score(values):
    values = np.asarray(values)
    return 1 - np.log10(values) / np.log10(50000)


def format_mean_std(values, digits=1):
    values = pd.Series(values)
    return f"{values.mean():.{digits}f}±{values.std():.{digits}f}"


def build_summary_table(df, value_label, group_col=None, digits=1):
    summary_source = df.groupby(group_col).mean(numeric_only=True) if group_col else df
    table = pd.DataFrame(
        {
            'Avg percentile': [
                format_mean_std(summary_source['avg_per_rand'], digits),
                format_mean_std(summary_source['avg_per_guided'], digits),
            ],
            'Top percentile': [
                format_mean_std(summary_source['max_per_rand'], digits),
                format_mean_std(summary_source['max_per_guided'], digits),
            ],
            f'Avg {value_label}': [
                format_mean_std(summary_source['avg_rand'], 2),
                format_mean_std(summary_source['avg_guided'], 2),
            ],
            f'Top {value_label}': [
                format_mean_std(summary_source['max_rand'], 2),
                format_mean_std(summary_source['max_guided'], 2),
            ],
        },
        index=['Random', 'Guided'],
    )
    return table


def objective_paths(objective):
    base = MODEL_DIR / objective
    return {
        'peptide_vae': base / 'p_vae.pt',
        'allele_vae': base / 'a_vae.pt',
        'gp': base / 'gp.pt',
    }


def clear_gp_cache(gp_model):
    gp_model.eval()
    gp_model.likelihood.eval()
    for obj in (gp_model, gp_model.likelihood, gp_model.variational_strategy):
        if hasattr(obj, '_clear_cache'):
            obj._clear_cache()


## Experimental model

Load the validation predictions, assemble the GP inputs, and evaluate random search against guided search across alleles.


In [ ]:
BATCH_SIZE = 8192//4
ALLELE_BATCH_SIZE = 512
TRAIN_PATH = DATA_DIR / 'train_pairs.csv'
VAL_PATH = DATA_DIR / 'val_pairs.csv'
ALLELE_TRAIN_PATH = DATA_DIR / 'train_allele.csv'
ALLELE_VAL_PATH = DATA_DIR / 'val_allele.csv'
OBJECTIVE = 'Experimental'

datamodule = PairsDataModule(
    BATCH_SIZE,
    OBJECTIVE,
    str(TRAIN_PATH),
    str(VAL_PATH),
    ALLELE_BATCH_SIZE,
    str(ALLELE_TRAIN_PATH),
    str(ALLELE_VAL_PATH),
)

paths = objective_paths(OBJECTIVE)
model = JointTraining(
    dataset=datamodule.pair_train,
    trans_vae_path=str(paths['peptide_vae']),
    cnn_vae_path=str(paths['allele_vae']),
    hidden_size=(64, 16),
    ystar=0.65,
)
model.gp.load_state_dict(torch.load(paths['gp'], map_location='cpu'))

trainer = pl.Trainer(strategy='auto')
predictions = trainer.predict(model, datamodule=datamodule, return_predictions=True)

gp_input = np.concatenate([batch[0] for batch in predictions], axis=0)
sim = np.concatenate([batch[1] for batch in predictions], axis=0)
allele_ids_np = np.concatenate([batch[2] for batch in predictions], axis=0)
affinity_np = np.concatenate([batch[3] for batch in predictions], axis=0)
qualitative_mask = np.concatenate([batch[4] for batch in predictions], axis=0).astype(bool)

gp = model.gp.to(DEVICE)
clear_gp_cache(gp)


In [ ]:
def evaluate_one_allele(
    allele_id,
    allele_mask,
    gp_model,
    gp_features,
    similarities,
    affinities,
    random_k=20,
    elite_frac=0.10,
):
    reference_affinity = affinities[allele_mask]
    current_gp = torch.as_tensor(gp_features[allele_mask], dtype=torch.float32, device=DEVICE)
    current_sim = similarities[allele_mask]
    current_affinity = affinities[allele_mask]

    sample_size = min(random_k, len(current_affinity))
    if sample_size == 0:
        return None

    random_sample = np.random.choice(current_affinity, size=sample_size, replace=False)
    elite_count = max(sample_size, int(np.ceil(len(current_sim) * elite_frac)))
    elite_idx = current_sim.argsort()[-elite_count:]
    elite_gp = current_gp[elite_idx]
    elite_affinity = current_affinity[elite_idx]
    guided_k = min(sample_size, len(elite_affinity))

    with torch.no_grad():
        _, _, selected_idx = TS(elite_gp, gp_model, guided_k, return_id=True)
        selected_idx = selected_idx[:, 0].flatten().detach().cpu().numpy()

    guided_sample = elite_affinity[selected_idx]
    return {
        'allele': allele_id,
        'avg_rand': np.mean(random_sample),
        'max_rand': np.max(random_sample),
        'avg_per_rand': percentile_mean(reference_affinity, random_sample),
        'max_per_rand': percentileofscore(reference_affinity, np.max(random_sample), 'rank'),
        'avg_guided': np.mean(guided_sample),
        'max_guided': np.max(guided_sample),
        'avg_per_guided': percentile_mean(reference_affinity, guided_sample),
        'max_per_guided': percentileofscore(reference_affinity, np.max(guided_sample), 'rank'),
    }


results = []
unique_alleles = np.unique(allele_ids_np)

for seed in tqdm(range(20,30)):
    set_seed(seed)
    for allele_id in unique_alleles:
        row = evaluate_one_allele(
            allele_id=allele_id,
            allele_mask=allele_ids_np == allele_id,
            gp_model=gp,
            gp_features=gp_input,
            similarities=sim,
            affinities=affinity_np,
        )
        if row is None:
            continue
        row['seed'] = seed
        results.append(row)

experimental_results = pd.DataFrame(results)
display(build_summary_table(experimental_results, value_label='Experimental', group_col='seed'))


## Binding affinity model

Load the BA models, generate candidates per allele, and compare random peptides with guided candidates using MHCflurry affinity scores.


In [ ]:
TRAIN_PATH = DATA_DIR / 'train_pairs.csv'
VAL_PATH = DATA_DIR / 'val_pairs.csv'
ALLELE_TRAIN_PATH = DATA_DIR / 'train_allele.csv'
ALLELE_VAL_PATH = DATA_DIR / 'val_allele.csv'
OBJECTIVE = 'BA'
paths = objective_paths(OBJECTIVE)

val_pairs = pd.read_csv(VAL_PATH)
train_pairs = pd.read_csv(TRAIN_PATH)
alleles = val_pairs.drop_duplicates(['allele']).loc[:, ['allele', 'pseudosequence']].reset_index(drop=True)
benchmark = val_pairs[val_pairs.measurement_inequality == '='].reset_index(drop=True)
benchmark_by_allele = {allele: df for allele, df in benchmark.groupby('allele')}

mhc = mhcflurry.Class1PresentationPredictor.load()
allele_dataset = AlleleDataset(alleles)
peptide_dataset = PeptideDataset(train_pairs)

a_vae = InfoCNNVAE2(allele_dataset, 64)
a_vae.load_state_dict(torch.load(paths['allele_vae'], map_location='cpu'))
a_vae = a_vae.to(DEVICE).eval()

p_vae = InfoTransformerVAE(peptide_dataset, 64)
p_vae.load_state_dict(torch.load(paths['peptide_vae'], map_location='cpu'))
p_vae = p_vae.to(DEVICE).eval()

likelihood = gpytorch.likelihoods.GaussianLikelihood()
gp = GPModelDKLExtended(
    torch.rand(1000, 128, device=DEVICE),
    likelihood,
    hidden_dims=(64, 16),
).to(DEVICE)
gp.load_state_dict(torch.load(paths['gp'], map_location='cpu'))
clear_gp_cache(gp)


In [ ]:
NUM_ROUNDS = 10
BATCH_SIZE = 20


def evaluate_generated_peptides(allele, guided_peptides, random_peptides):
    allele_data = benchmark_by_allele.get(allele)
    if allele_data is None or len(guided_peptides) == 0:
        return None

    with suppress_stdout():
        guided_aff = mhc.predict(guided_peptides, [allele]).affinity.values
        random_aff = mhc.predict(random_peptides, [allele]).affinity.values

    ref_aff = affinity_to_score(allele_data.affinity.values)
    guided_aff = affinity_to_score(guided_aff)
    random_aff = affinity_to_score(random_aff)

    return {
        'allele': allele,
        'avg_rand': np.mean(random_aff),
        'max_rand': np.max(random_aff),
        'avg_per_rand': percentile_mean(ref_aff, random_aff),
        'max_per_rand': percentileofscore(ref_aff, np.max(random_aff), 'rank'),
        'avg_guided': np.mean(guided_aff),
        'max_guided': np.max(guided_aff),
        'avg_per_guided': percentile_mean(ref_aff, guided_aff),
        'max_per_guided': percentileofscore(ref_aff, np.max(guided_aff), 'rank'),
    }


results = []

for round_idx in tqdm(range(NUM_ROUNDS)):
    set_seed(round_idx)
    for _, embeds, allele in allele_dataset:
        allele_latent = a_vae.encode(embeds.unsqueeze(0).to(DEVICE))[0]
        init_z = torch.cat((allele_latent, allele_latent), dim=1)

        state = TurboState(dim=init_z.shape[-1], batch_size=BATCH_SIZE, length=0.2)
        candidate_z, _, _ = generate_batch(
            state=state,
            model=gp,
            X=init_z,
            Y=torch.zeros(1, device=DEVICE),
            batch_size=BATCH_SIZE,
            device=DEVICE,
        )

        peptide_z = candidate_z[:, :64].reshape(-1, 1, p_vae.d_model).to(DEVICE)
        sampled_tokens = p_vae.sample(z=peptide_z)
        guided_peptides = [p_vae.dataset.decode(sampled_tokens[i]) for i in range(sampled_tokens.size(0))]
        guided_peptides = [peptide for peptide in guided_peptides if 5 <= len(peptide) <= 15]

        random_peptides = train_pairs.sample(n=BATCH_SIZE).peptide.tolist()
        row = evaluate_generated_peptides(allele, guided_peptides, random_peptides)
        if row is not None:
            row['seed'] = round_idx
            results.append(row)

ba_results = pd.DataFrame(results)
display(build_summary_table(ba_results, value_label='BA score',group_col='seed'))

